# Protein Sequence Alignment with OTalign

This notebook demonstrates how to load a trained OTalign model and align two protein sequences using the Unbalanced Optimal Transport (UOT) framework.

## Overview

The alignment pipeline consists of four stages:
1. **Load the PLM** — Load a protein language model (with optional LoRA fine-tuning).
2. **Encode sequences** — Generate per-residue embeddings using the PLM.
3. **Compute the transport plan** — Use the UOT Sinkhorn algorithm to find an optimal soft mapping between residues.
4. **Extract hard alignment** — Convert the soft transport plan into a discrete gapped alignment using dynamic programming with position-specific gap penalties.

**Note:** This notebook requires a trained model checkpoint. Set the `checkpoint_path` variable in Section 2 to a valid checkpoint path. Without it, the code will fall back to the base model.

## 1. Import Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml

# OTalign core components
from otalign.align.cost import pairwise_cosine
from otalign.align.uot_alignment import hard_alignment_from_transport
from otalign.functional.sinkhorn_uot import unbalanced_sinkhorn
from otalign.models.plm_adaptors import get_plm_adaptor_and_configs
from otalign.utils.checkpointing import load_peft_model_from_checkpoint
from otalign.utils.display import print_alignment
from otalign.viz import plot_plan_with_domains

## 2. Configuration and Model Loading

Specify the training config file and the model checkpoint path. **Update `checkpoint_path` to point to your trained model.**

The PLM adaptor handles tokenization and embedding extraction. If a LoRA checkpoint is found, it is loaded on top of the base model.

In [ ]:
# Path to the training configuration file
config_path = "configs/train_config.yaml"

# Path to the model checkpoint (UPDATE THIS to your actual checkpoint path)
# Example: "work/checkpoints/esm1b-lora-finetune-2/checkpoint-epoch-3"
# You can also use a base model name like "AnkhCL" (without fine-tuning)
checkpoint_path = "work/checkpoints/esm1b-lora-finetune-2/checkpoint-epoch-3"

with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the PLM adaptor and base model
plm_adaptor, _, _ = get_plm_adaptor_and_configs(config["model_name"], for_masked_lm=True)
model = plm_adaptor.model

# Load LoRA weights from checkpoint if available
if Path(checkpoint_path).exists():
    lora_model = load_peft_model_from_checkpoint(model, checkpoint_path)
    lora_model.to(device)
    lora_model.eval()
    plm_adaptor.model = lora_model
    print(f"Model loaded from {checkpoint_path}")
else:
    print(f"[Warning] Checkpoint not found: {checkpoint_path}")
    print("Falling back to the base model (results may differ from fine-tuned model).")
    lora_model = None
    model.to(device)
    model.eval()

## 3. Define Protein Sequences

Enter two protein sequences to align. These can be any valid amino acid sequences.

In [ ]:
# Example protein sequences for alignment
seq1 = "AGLPVIMCLKSNNHQKYLRYQSDNIQQYGLLQFSADKILDPLAQFEVEPSKTY"
seq2 = "DGLVHIKSRYTNKYLVRWSPNHYWITASANEPDENKSNWACTLFKPLYVEEGN"

## 4. Generate Residue Embeddings and Cost Matrix

Each sequence is encoded through the PLM to produce per-residue embeddings.
The **cost matrix** is computed as the pairwise cosine distance between all residue
pairs across the two sequences: `C[i,j] = 1 - cosine_similarity(emb1[i], emb2[j])`.

Low cost values indicate residues with similar PLM representations — likely structural or functional homologs.

In [ ]:
def get_embeddings(sequence, adaptor, device):
    """Encode a single protein sequence and return its residue embeddings."""
    if adaptor.model is None:
        # Return random embeddings as a fallback if model is not loaded
        return torch.randn(1, len(sequence), 1280).to(device), torch.tensor([len(sequence)]).to(device)

    with torch.no_grad():
        emb_out = adaptor.encode([sequence], device=device, disable_grad=True)
        return emb_out.residue_embeddings, torch.tensor([len(sequence)]).to(device)


emb1, len1 = get_embeddings(seq1, plm_adaptor, device)
emb2, len2 = get_embeddings(seq2, plm_adaptor, device)

# Compute the cosine distance cost matrix between residue embeddings
cost_matrix = pairwise_cosine(emb1, emb2)

print(f"Sequence 1 length: {len1.item()}")
print(f"Sequence 2 length: {len2.item()}")
print(f"Cost matrix shape: {cost_matrix.shape}")

## 5. Compute the Unbalanced Optimal Transport Plan

The UOT Sinkhorn algorithm finds an optimal transport plan between the two sets of
residue embeddings. Key parameters:

- **`reg`** (entropy regularization): Controls the smoothness of the plan. Lower = sharper.
- **`reg_m`** (marginal relaxation): Controls how much mass can be "destroyed" — critical
  for sequences of different lengths or with unaligned regions.
- **`num_iter`**: Number of Sinkhorn iterations (1000 is usually sufficient).

The transport plan `P[i,j]` represents the amount of mass transported from query
residue `i` to template residue `j`. The dual variables `u` and `v` (scaling vectors)
encode per-residue alignment confidence and are used to derive adaptive gap penalties.

In [ ]:
# Override reg_m if needed (controls marginal relaxation)
config["uot"]["reg_m"] = 1.0

B, N, M = cost_matrix.shape
lens1, lens2 = len1.to(device), len2.to(device)

# Create masks and uniform marginal distributions a, b
mask1 = torch.arange(N, device=device)[None, :] < lens1[:, None]
mask2 = torch.arange(M, device=device)[None, :] < lens2[:, None]
a = mask1.float() / lens1[:, None].clamp(min=1).float()
b = mask2.float() / lens2[:, None].clamp(min=1).float()

# Set cost outside valid regions to a large value
cost_matrix[~(mask1[:, :, None] * mask2[:, None, :])] = 1e6

# Run the UOT Sinkhorn algorithm to obtain the transport plan
transport_plan, u, v = unbalanced_sinkhorn(
    cost_matrix,
    a,
    b,
    config["uot"]["num_iter"],
    config["uot"]["reg"],
    config["uot"]["reg_m"],
    config["uot"]["reg_m"],
    mask_a=mask1,
    mask_b=mask2,
)

print(f"Transport plan shape: {transport_plan.shape}")

## 6. Extract Hard Alignment and Visualize

The transport plan is converted to a discrete alignment using dynamic programming.
The function `hard_alignment_from_transport` computes:

- **Match scores** from Pointwise Mutual Information (PMI) of the transport plan.
- **Position-specific gap penalties** from the marginal mass and dual potentials (`f`, `g`).

The resulting alignment is displayed as a colored text alignment and as a transport
plan heatmap. In the heatmap:
- Bright regions along the diagonal indicate confidently aligned residue pairs.
- The side bars show the dual potentials (log-space) — negative values correspond to
  residues more likely to be gapped.

In [ ]:
# Convert transport plan and duals to numpy
plan = transport_plan[0].cpu().numpy()
f = np.log(u[0].cpu().numpy())  # log-space dual potential for query
g = np.log(v[0].cpu().numpy())  # log-space dual potential for template
s = np.log(plan)  # log-space transport plan (for visualization)

# Run dynamic programming to get the hard alignment
# mode="glocal" allows free terminal gaps on both ends
aln_dict = hard_alignment_from_transport(plan, mode="glocal", score_scale=0.5, meta=True)
matches = np.array([(x - 1, y - 1) for x, y, o in aln_dict["path"] if o == "M"], dtype=np.int64)

# Display the text alignment with color-coded match scores
print_alignment(seq1, seq2, aln_dict["path"], score=s, cmap=plt.get_cmap("terrain_r"), display_mode="global_view")

# Visualize the transport plan heatmap with dual potential side bars
fig, ax = plot_plan_with_domains(
    plan,
    f=f - np.median(f),
    g=g - np.median(g),
    cmap="terrain_r",
    colorbar=None,
)
fig.set_figwidth(6)
fig.set_figheight(6)
ax.set_aspect("equal")
fig.tight_layout()
fig.show()
fig.savefig("plan.svg")